In [2]:
# Clear model outputs before %run so figures cannot silently reuse stale notebook globals.
for _name in list(globals()):
    if (_name.startswith(("df_", "res_", "finance_", "kpi_table_", "deltas_", "capex_", "opex_", "spb_"))
            or _name == "RESULTS"):
        globals().pop(_name, None)

%run "./Clean CRAH Financial Comparison Model.ipynb"


Loaded source — Phoenix: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\PHOENIX_DataCenter_Profile.xlsx
Loaded source — Fairbanks: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\FAIRBANKS_DataCenter_Profile.xlsx
Loaded rows — Phoenix: 8760 Fairbanks: 8760
Qserver first hour — Phoenix: 1342.595476084541 Fairbanks: 1342.595476084541

=== PHOENIX — Hourly (first 6 of 8760) ===
                 mode        IT_kW  Twb_out_C  P_sys_mag_kW  P_sys_cen_kW  Δ_kW (cen - mag)
0       MODE3_CHILLER  1342.595476      11.18    100.594686    114.760547         14.165861
1       MODE3_CHILLER  1342.685053       9.79     95.943290    107.190236         11.246946
2       MODE3_CHILLER  1340.395705       8.49     91.698479    100.542454          8.843975
3  MODE2_PARTIAL_FREE  1338.474414       7.79     85.843891     93.438577          7.594686
4       MODE3_CHILLER  1339.993680       8.03     90.290431     98.355683          8.065253
5  MODE2_PARTIAL_FREE  1335.405900       6.58  

In [3]:
%run "./Fin model with ATES.ipynb"

# The ATES notebook now publishes per-site outputs used by later plotting cells.
required_outputs = ["res_cen_phx", "res_mag_phx", "res_amg_phx",
                    "res_cen_fb", "res_mag_fb", "res_amg_fb",
                    "finance_phx", "finance_fb"]
missing_outputs = [name for name in required_outputs if name not in globals()]
if missing_outputs:
    raise RuntimeError(f"Missing fresh model outputs after ATES run: {missing_outputs}")



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
1.0.0
Loaded source — Phoenix: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\PHOENIX_DataCenter_Profile.xlsx
Loaded source — Fairbanks: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\FAIRBANKS_DataCenter_Profile.xlsx
Loaded rows — Phoenix: 8760 Fairbanks: 8760
Qserver first hour — Phoenix: 1342.595476084541 Fairbanks: 1342.595476084541

=== PHOENIX — Hourly (first 6 of 8760) ===
                 mode        IT_kW  Twb_out_C  P_sys_mag_kW  P_sys_cen_kW  Δ_kW (cen - mag)
0       MODE3_CHILLER  1342.595476      11.18    100.594686    114.760547         14.165861
1       MODE3_CHILLER  1342.685053       9.79     95.943290    107.190236         11.246946
2       MODE3_CHILLER  1340.395705       8.49     91.698479    100.542454          8.843975
3  MODE2_PARTIAL_FREE  1338.474414       7.79     85.843891     93.438577          7.594686
4       MODE3_CHILLER  1339.993680       8.03     90.290431     98.

In [3]:
# === Fig 4.1 — Generate & SAVE (Phoenix & Fairbanks) with confirmations ===

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------- 0) OUTPUT FOLDER (verify write access) --------
FIG_DIR = r"C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures"
os.makedirs(FIG_DIR, exist_ok=True)
print("[ok] FIG_DIR:", FIG_DIR)
# quick write test
_test_path = os.path.join(FIG_DIR, "_write_test.tmp")
with open(_test_path, "w", encoding="utf-8") as f:
    f.write("ok")
os.remove(_test_path)

# -------- 1) WET-BULB PROXY (no new inputs) + RECOMPUTE RESULTS --------
ADIABATIC_RELIEF_K = {"Phoenix": 10.0, "Fairbanks": 3.0}  # adjust if desired

def apply_pseudo_wetbulb(df: pd.DataFrame, site_name: str, tower_Tmin_C: float = 0.0) -> pd.DataFrame:
    df = df.copy()
    gamma = ADIABATIC_RELIEF_K[site_name]
    twb = df["Tdb_out_C"].astype(float) - gamma
    twb = np.maximum(twb, tower_Tmin_C)
    twb = np.minimum(twb, df["Tdb_out_C"].astype(float))
    df["Twb_out_C"] = twb
    return df

# Ensure the base hourly frames exist
assert "df_phoenix" in globals() and "df_fairbanks" in globals(), "Load df_phoenix/df_fairbanks first."

df_phx_twb = apply_pseudo_wetbulb(df_phoenix,   "Phoenix",   tower_Tmin_C=0.0)
df_fb_twb  = apply_pseudo_wetbulb(df_fairbanks, "Fairbanks", tower_Tmin_C=0.0)

# Ensure compare_plans, p_mag, p_cen, SITES exist
assert "compare_plans" in globals() and "p_mag" in globals() and "p_cen" in globals() and "SITES" in globals(), \
       "Run the model setup cells that define compare_plans, p_mag, p_cen, SITES."

res_mag_phx, res_cen_phx, _, _ = compare_plans(
    df_phx_twb, p_mag, p_cen, elec_price=SITES["Phoenix"]["energy_price_usd_per_kWh"], dt_hours=1.0
)
res_mag_fb,  res_cen_fb,  _, _ = compare_plans(
    df_fb_twb,  p_mag, p_cen, elec_price=SITES["Fairbanks"]["energy_price_usd_per_kWh"], dt_hours=1.0
)

# -------- 2) PLOT HELPERS --------
MODE_COLORS = {
    "MODE1_FULL_FREE":   "#90BE6D",
    "MODE2_PARTIAL_FREE":"#F9C74F",
    "MODE3_CHILLER":     "#F94144",
}

def _month_series(df: pd.DataFrame) -> pd.Series:
    if "DateTime" in df.columns and not df["DateTime"].isna().all():
        return pd.to_datetime(df["DateTime"], errors="coerce").dt.month.fillna(1).astype(int)
    idx = pd.date_range("2022-01-01", periods=len(df), freq="h")  # no FutureWarning
    return pd.Series(idx.month, index=df.index)

def _mode_month_counts(res_df: pd.DataFrame) -> pd.DataFrame:
    months = _month_series(res_df)
    tmp = pd.DataFrame({"month": months, "mode": res_df["mode"].values})
    counts = (tmp.groupby(["month", "mode"]).size()
              .unstack(fill_value=0)
              .reindex(index=range(1,13), fill_value=0))
    for m in ["MODE1_FULL_FREE","MODE2_PARTIAL_FREE","MODE3_CHILLER"]:
        if m not in counts.columns: counts[m] = 0
    counts = counts[["MODE1_FULL_FREE","MODE2_PARTIAL_FREE","MODE3_CHILLER"]]
    counts.index = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    return counts

def plot_mode_duration(city_name: str, res_df: pd.DataFrame, savepath: str,
                       bar_width: float = 0.45, figsize=(9, 4.6), headroom: int = 40):
    counts = _mode_month_counts(res_df)
    x = np.arange(len(counts))
    m1 = counts["MODE1_FULL_FREE"].values
    m2 = counts["MODE2_PARTIAL_FREE"].values
    m3 = counts["MODE3_CHILLER"].values
    total = m1 + m2 + m3

    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(x, m1, width=bar_width, color=MODE_COLORS["MODE1_FULL_FREE"],  edgecolor="black", label="Mode 1 - Free cooling without chiller")
    ax.bar(x, m2, width=bar_width, bottom=m1,  color=MODE_COLORS["MODE2_PARTIAL_FREE"], edgecolor="black", label="Mode 2 - Partial cooling with chiller")
    ax.bar(x, m3, width=bar_width, bottom=m1+m2, color=MODE_COLORS["MODE3_CHILLER"],   edgecolor="black", label="Mode 3 - Chiller provides full cooling")

    ax.set_xticks(x); ax.set_xticklabels(list(counts.index))
    ax.set_xlabel("Month")
    ax.set_ylabel("Hours")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.set_ylim(0, float(total.max()) + headroom)  # avoid clipping
    # The caption is handled in the manuscript; keep the x-axis label as the plotted variable.
    # Add figure label below x-axis (caption style)
    #ax.set_xlabel(f"Figure 4.{1 if city_name=='Phoenix' else 2}: {city_name} — Monthly Operating Mode Duration",
              #labelpad=15, fontsize=11)



    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=3, frameon=False)
    plt.tight_layout(); plt.subplots_adjust(top=0.82)

    fig.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[saved] {savepath}")

# -------- 3) SAVE BOTH FIGURES --------
phx_path = os.path.join(FIG_DIR, "Fig4_1_Phoenix_ModeDuration.png")
fb_path  = os.path.join(FIG_DIR, "Fig4_2_Fairbanks_ModeDuration.png")

plot_mode_duration("Phoenix",   res_mag_phx, savepath=phx_path)
plot_mode_duration("Fairbanks", res_mag_fb,  savepath=fb_path)

# -------- 4) SHOW FOLDER CONTENTS --------
print("[dir] Result Figures contains:")
for name in os.listdir(FIG_DIR):
    if name.lower().endswith((".png",".jpg",".jpeg",".svg",".pdf")):
        print("  -", name)


[ok] FIG_DIR: C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures
[saved] C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures\Fig4_1_Phoenix_ModeDuration.png
[saved] C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures\Fig4_2_Fairbanks_ModeDuration.png
[dir] Result Figures contains:
  - Fig4_1_Phoenix_ModeDuration.png
  - Fig4_2_Fairbanks_ModeDuration.png
  - Fig4_3_Phoenix_Energy.png
  - Fig4_4_Fairbanks_Energy.png
  - Fig4_5_ModeDuration_ByCity.png
  - Fig4_6_Phoenix_DynamicPayback.png
  - Fig4_TotalCostStack.png
  - Fig4_X_Cost_With_SPB.png


In [4]:
# === Fig 4.3 & 4.4 — Annual energy consumption by configuration (Phoenix & Fairbanks)
# Uses ONLY objects already in memory from your %run notebooks. No recomputation, no hardcoding.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Output folder ----------
FIG_DIR = r"C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ---------- Visual style ----------
USE_MWH    = True       # False → keep kWh
BAR_WIDTH  = 0.45
FIGSIZE    = (9, 4.6)
HEADROOM   = 40
CAPTION_PAD = 15
CLR = {
    "Centrifugal (Baseline)": "#F9C74F",
    "Magnetic":               "#90BE6D",
    "MBC+ATES":          "#4DABF7",
}

def _resolve_result(site: str, plan: str):
    """
    Return the first matching results object found in globals() for (site, plan).
    We DO NOT recompute anything here.
    """
    name_sets = {
        ("Phoenix","cen"):  ["res_cen_phx", "res_cenPhoenix", "res_cen"],
        ("Phoenix","mag"):  ["res_mag_phx", "res_magPhoenix", "res_mag"],
        ("Phoenix","ates"): ["res_ates_phx", "res_amg_phx", "res_amgPhoenix", "res_amg", "res_ates"],

        ("Fairbanks","cen"):  ["res_cen_fb", "res_cenFairbanks", "res_cen"],
        ("Fairbanks","mag"):  ["res_mag_fb", "res_magFairbanks", "res_mag"],
        ("Fairbanks","ates"): ["res_ates_fb", "res_amg_fb", "res_amgFairbanks", "res_amg", "res_ates"],
    }
    for nm in name_sets.get((site, plan), []):
        if nm in globals():
            return globals()[nm]
    return None

def _annual_kwh(res_df: pd.DataFrame) -> float:
    # 1-hour timesteps → sum of P_sys_kW is annual kWh
    return float(pd.Series(res_df["P_sys_kW"]).sum())

def _pue_prime_avg(res_df: pd.DataFrame) -> float | None:
    # PUE′ = 1 + P_sys/IT, with IT = Qsum/1.10 (your 10% facility overhead convention)
    try:
        it = np.asarray(res_df["Qsum_kW"], dtype=float) / 1.10
        ps = np.asarray(res_df["P_sys_kW"], dtype=float)
        pue = 1.0 + np.divide(ps, it, out=np.full_like(ps, np.nan), where=it>0)
        v = float(np.nanmean(pue))
        return v if np.isfinite(v) else None
    except Exception:
        return None

def _build_table(site: str) -> pd.DataFrame:
    res_cen  = _resolve_result(site, "cen")
    res_mag  = _resolve_result(site, "mag")
    res_ates = _resolve_result(site, "ates")   # may be None

    if res_cen is None or res_mag is None:
        raise RuntimeError(f"{site}: missing res_cen/res_mag in memory. Re-run your CRAH %run first.")

    rows = [("Centrifugal (Baseline)", res_cen),
            ("Magnetic",               res_mag)]

    # Only include ATES if we truly found a site-matching object in memory
    if res_ates is not None:
        rows.append(("MBC+ATES",  res_ates))
    else:
        print(f"[info] {site}: ATES results not found in memory — plotting only baseline & magnetic "
              "(avoids mixing inconsistent baselines).")

    df = pd.DataFrame({
        "Plan": [name for name,_ in rows],
        "kWh":  [_annual_kwh(obj) for _,obj in rows],
        "PUEp": [_pue_prime_avg(obj) for _,obj in rows],
    })

    df["Value"] = df["kWh"]/1_000.0 if USE_MWH else df["kWh"]
    return df

def _plot_energy(site: str, df_site: pd.DataFrame, savepath: str):
    # enforce consistent plan order with whatever is present
    order = [p for p in ["Centrifugal (Baseline)", "Magnetic", "MBC+ATES"]
             if p in df_site["Plan"].tolist()]
    dfp = df_site.set_index("Plan").loc[order].reset_index()

    units  = "MWh" if USE_MWH else "kWh"
    ylabel = f"Annual energy consumption ({units})"

    # show PUE′ overlay only if every bar has a valid value
    pue_ok = (len(dfp) >= 2) and dfp["PUEp"].notna().all()

    # console sanity print
    print(f"\n[{site}]")
    for _, r in dfp.iterrows():
        print(f"  {r['Plan']}: {r['Value']:,.1f} {units} | PUE′ avg: "
              f"{'—' if pd.isna(r['PUEp']) else f'{r['PUEp']:.3f}'}")

    x = np.arange(len(dfp))
    fig, ax1 = plt.subplots(figsize=FIGSIZE)

    ax1.bar(x, dfp["Value"], width=BAR_WIDTH,
            color=[CLR[p] for p in dfp["Plan"]],
            edgecolor="black")
    ax1.set_xticks(x); ax1.set_xticklabels(dfp["Plan"])
    ax1.set_ylabel(ylabel)
    ax1.grid(axis="y", linestyle="--", alpha=0.35)
    ymax = float(dfp["Value"].max()) if len(dfp) else 0.0
    ax1.set_ylim(0, ymax + HEADROOM)

    # caption-style label (no figure number inside)
    ax1.set_xlabel("Cooling configuration")

    if pue_ok:
        ax2 = ax1.twinx()
        ax2.plot(x, dfp["PUEp"], marker="o", linewidth=1.5, label="PUE′ (avg)")
        ax2.set_ylabel("Cooling-only PUE′ (—)")
        ax2.set_ylim(0, max(1.05, float(dfp["PUEp"].max())*1.15))
        ax2.legend(loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=1, frameon=False)

    plt.tight_layout()
    plt.subplots_adjust(top=0.88, bottom=0.20)
    fig.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[saved] {savepath}")

# -------- Build tables from whatever your %run notebooks left in memory --------
df_phx = _build_table("Phoenix")
df_fb  = _build_table("Fairbanks")

# -------- Save plots (no figure numbers in the image) --------
_plot_energy("Phoenix",   df_phx, savepath=os.path.join(FIG_DIR, "Fig4_3_Phoenix_Energy.png"))
_plot_energy("Fairbanks", df_fb,  savepath=os.path.join(FIG_DIR, "Fig4_4_Fairbanks_Energy.png"))



[Phoenix]
  Centrifugal (Baseline): 1,302.5 MWh | PUE′ avg: 1.105
  Magnetic: 1,001.3 MWh | PUE′ avg: 1.081
  MBC+ATES: 1,704.5 MWh | PUE′ avg: 1.138
[saved] C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures\Fig4_3_Phoenix_Energy.png

[Fairbanks]
  Centrifugal (Baseline): 402.7 MWh | PUE′ avg: 1.033
  Magnetic: 342.7 MWh | PUE′ avg: 1.028
  MBC+ATES: 437.4 MWh | PUE′ avg: 1.036
[saved] C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures\Fig4_4_Fairbanks_Energy.png


In [5]:
# === LEGACY/DISABLED Fig 4.3 & 4.4 hard-coded energy cell ===
# Disabled because it overwrites the fresh model-based figures above with old KPI values.
# Third bar is labelled "MBC+ATES" (was ATES+Magnetic).

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Output folder ----------
FIG_DIR = r"C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ---------- Visual style ----------
USE_MWH     = True       # False → keep kWh
BAR_WIDTH   = 0.45
FIGSIZE     = (9, 4.6)
HEADROOM    = 40
CAPTION_PAD = 15

CLR = {
    "Centrifugal Chiller (as baseline)": "#F9C74F",
    "MBC":               "#90BE6D",
    "MBC+ATES":               "#4DABF7",
}

# ---------- 1) Hard data from your Phoenix / Fairbanks KPI tables ----------

# Phoenix — Annual KPIs & CapEx ===
# Plan            kWh/yr         PUE' avg
# Centrifugal     1.968709e+06   1.196958
# Magnetic        1.411402e+06   1.140226
# ATES+Magnetic   1.411140e+06   1.140268

df_phx = pd.DataFrame({
    "Plan": ["Centrifugal Chiller (as baseline)", "MBC", "MBC+ATES"],
    "kWh":  [1.968709e6,               1.411402e6,  1.411140e6],
    "PUEp": [1.196958,                 1.140226,    1.140268],
})

# Fairbanks — Annual KPIs & CapEx ===
# Plan            kWh/yr         PUE' avg
# Centrifugal     352097.153659  1.042257
# Magnetic        278587.131745  1.033272
# ATES+Magnetic   291889.999989  1.035027

df_fb = pd.DataFrame({
    "Plan": ["Centrifugal Chiller (as baseline)", "MBC", "MBC+ATES"],
    "kWh":  [352097.153659,            278587.131745, 291889.999989],
    "PUEp": [1.042257,                 1.033272,      1.035027],
})

# convert to plotting value (MWh or kWh)
for df in (df_phx, df_fb):
    df["Value"] = df["kWh"] / 1_000.0 if USE_MWH else df["kWh"]

# ---------- 2) Plot helper ----------

def _plot_energy(site: str, df_site: pd.DataFrame, savepath: str):
    # enforce consistent plan order
    order = ["Centrifugal Chiller (as baseline)", "MBC", "MBC+ATES"]
    dfp = df_site.set_index("Plan").loc[order].reset_index()

    units  = "MWh" if USE_MWH else "kWh"
    ylabel = f"Annual energy consumption ({units})"

    # show PUE′ overlay only if every bar has a valid value
    pue_ok = dfp["PUEp"].notna().all()

    # console sanity print
    print(f"\n[{site}]")
    for _, r in dfp.iterrows():
        print(
            f"  {r['Plan']}: {r['Value']:,.1f} {units} | "
            f"PUE′ avg: {r['PUEp']:.3f}"
        )

    x = np.arange(len(dfp))
    fig, ax1 = plt.subplots(figsize=FIGSIZE)

    ax1.bar(
        x,
        dfp["Value"],
        width=BAR_WIDTH,
        color=[CLR[p] for p in dfp["Plan"]],
        edgecolor="black",
    )
    ax1.set_xticks(x)
    ax1.set_xticklabels(dfp["Plan"])
    ax1.set_ylabel(ylabel)
    ax1.grid(axis="y", linestyle="--", alpha=0.35)

    ymax = float(dfp["Value"].max()) if len(dfp) else 0.0
    ax1.set_ylim(0, ymax + HEADROOM)

    ax1.set_xlabel("Cooling configuration")
        fontsize=11,
    )

    if pue_ok:
        ax2 = ax1.twinx()
        ax2.plot(
            x,
            dfp["PUEp"],
            marker="o",
            linewidth=1.5,
            label="PUE′ (avg)",
        )
        ax2.set_ylabel("Cooling-only PUE′ (—)")
        ax2.set_ylim(0, max(1.05, float(dfp["PUEp"].max()) * 1.15))
        ax2.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, 1.12),
            ncol=1,
            frameon=False,
        )

    plt.tight_layout()
    plt.subplots_adjust(top=0.88, bottom=0.20)
    fig.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[saved] {savepath}")

# ---------- 3) Legacy hard-coded figure generation intentionally skipped ----------
print("[skip] Legacy hard-coded Fig4_3/Fig4_4 cell disabled; fresh model-based figures were saved above.")


[skip] Legacy hard-coded Fig4_3/Fig4_4 cell disabled; fresh model-based figures were saved above.


In [6]:
# === Fig 4.5 — Duration of Annual Operation by Mode (Phoenix & Fairbanks; Fig.10 style) ===
# Requires in memory: df_phoenix, df_fairbanks, p_mag, plan_power_timeseries

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Config (consistent visuals) ----------
FIG_DIR     = r"C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures"
os.makedirs(FIG_DIR, exist_ok=True)

FIGSIZE     = (9, 4.6)
BAR_WIDTH   = 0.55
CAPTION_PAD = 15

MODE_COLORS = {
    "MODE1_FULL_FREE":    "#90BE6D",  # green
    "MODE2_PARTIAL_FREE": "#F9C74F",  # yellow
    "MODE3_CHILLER":      "#F94144",  # red
}

# Same λ (adiabatic relief) as your monthly figures — applied **locally only**
ADIABATIC_RELIEF_K = {"Phoenix": 10.0, "Fairbanks": 3.0}

def _apply_pseudo_wetbulb_local(df_in: pd.DataFrame, site_name: str, tower_Tmin_C: float = 0.0) -> pd.DataFrame:
    df = df_in.copy()
    gamma = ADIABATIC_RELIEF_K[site_name]
    twb = df["Tdb_out_C"].astype(float) - gamma
    twb = np.maximum(twb, tower_Tmin_C)
    twb = np.minimum(twb, df["Tdb_out_C"].astype(float))
    df["Twb_out_C"] = twb
    return df

def _annual_mode_counts(df_city: pd.DataFrame, site_name: str) -> dict:
    """Return annual hours per mode using your engine; no global mutation."""
    df_local = _apply_pseudo_wetbulb_local(df_city, site_name, tower_Tmin_C=0.0)
    res = plan_power_timeseries(df_local, p_mag)  # modes same for cen/mag; use p_mag for convenience
    vc = res["mode"].value_counts()
    return {
        "MODE1_FULL_FREE":    int(vc.get("MODE1_FULL_FREE", 0)),
        "MODE2_PARTIAL_FREE": int(vc.get("MODE2_PARTIAL_FREE", 0)),
        "MODE3_CHILLER":      int(vc.get("MODE3_CHILLER", 0)),
    }

# --- Compute counts ---
phx = _annual_mode_counts(df_phoenix,   "Phoenix")
fbk = _annual_mode_counts(df_fairbanks, "Fairbanks")

cities = ["Phoenix", "Fairbanks"]
m1 = np.array([phx["MODE1_FULL_FREE"],    fbk["MODE1_FULL_FREE"]],    dtype=int)
m2 = np.array([phx["MODE2_PARTIAL_FREE"], fbk["MODE2_PARTIAL_FREE"]], dtype=int)
m3 = np.array([phx["MODE3_CHILLER"],      fbk["MODE3_CHILLER"]],      dtype=int)
tot = m1 + m2 + m3

# --- Plot (Fig.10 style) ---
x = np.arange(len(cities))
fig, ax = plt.subplots(figsize=FIGSIZE)

b1 = ax.bar(x, m1, width=BAR_WIDTH, color=MODE_COLORS["MODE1_FULL_FREE"],    edgecolor="black", label="Mode 1 - Free cooling without chiller")
b2 = ax.bar(x, m2, width=BAR_WIDTH, bottom=m1, color=MODE_COLORS["MODE2_PARTIAL_FREE"], edgecolor="black", label="Mode 2 - Partial cooling with chiller")
b3 = ax.bar(x, m3, width=BAR_WIDTH, bottom=m1+m2, color=MODE_COLORS["MODE3_CHILLER"],      edgecolor="black", label="Mode 3 - Chiller provides full cooling")

ax.set_xticks(x)
ax.set_xticklabels(cities)
ax.set_ylabel("Hours of operation")
ax.grid(axis="y", linestyle="--", alpha=0.35)

# Caption at bottom (no figure number in image)
ax.set_xlabel("City")

# --- Annotate segment hours & total per city ---
def _label_bar_segments(ax, x_pos, bottoms, heights, fmt="{:d}", min_px=14):
    """
    Draw integer labels centered on each colored segment if the segment is tall enough.
    min_px: minimum on-plot pixel height to show label (avoids clutter on tiny slices).
    """
    # Convert a height in data units to pixels to decide visibility
    trans = ax.transData.transform
    y0 = ax.get_ylim()[0]
    _, py0 = trans((0, y0))
    for i, (x0, yb, h) in enumerate(zip(x_pos, bottoms, heights)):
        if h <= 0:
            continue
        # pixel height of this segment
        _, py1 = trans((0, yb + h))
        _, pyb = trans((0, yb))
        if abs(py1 - pyb) >= min_px:
            ax.text(x0, yb + h/2, fmt.format(int(h)),
                    ha="center", va="center", fontsize=8, color="black")

# Segment labels
_label_bar_segments(ax, x, bottoms=np.zeros_like(m1), heights=m1)
_label_bar_segments(ax, x, bottoms=m1,                 heights=m2)
_label_bar_segments(ax, x, bottoms=m1+m2,              heights=m3)

# Totals on top
for xi, t in zip(x, tot):
    ax.text(xi, t + max(60, 0.01*tot.max()), f"{int(t)}",
            ha="center", va="bottom", fontsize=8)

# Legend (top center), consistent spacing
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=3, frameon=False)

plt.tight_layout()
plt.subplots_adjust(top=0.86, bottom=0.20)

outpath = os.path.join(FIG_DIR, "Fig4_5_ModeDuration_ByCity.png")
fig.savefig(outpath, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[saved] {outpath}  |  Totals (hrs): Phoenix={int(tot[0])}, Fairbanks={int(tot[1])}")


[saved] C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures\Fig4_5_ModeDuration_ByCity.png  |  Totals (hrs): Phoenix=8760, Fairbanks=8760


In [7]:
# # === Fig 4.6 & 4.7 — Dynamic Payback (Premium CapEx & NPV both in M$) + OPTIONAL GRAY BOXES ===
# # This replaces plot_dynamic_no_labels() with plot_dynamic_with_boxes(show_boxes=True/False).
# # Everything else (data extraction, file paths, colors, legend, caption) stays identical.

# import os, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
# from matplotlib.lines import Line2D

# FIG_DIR = r"C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures"
# os.makedirs(FIG_DIR, exist_ok=True)

# FIGSIZE = (9.0, 4.8)

# # === TUNE MARKER SIZES HERE ==================================================
# CAPEX_MARKER_SIZE = 100   # ⬅️ bigger circles for premium CapEx points
# NPV_MARKER_SIZE   = 3     # ⬅️ smaller diamonds for the NPV line markers
# # ============================================================================

# COLORS = {
#     "Mag vs Cen"      : "#90BE6D",  # green
#     "ATES+Mag vs Cen" : "#4DABF7",  # blue
# }

# def _find_col(df, *must, any_of=None):
#     for c in df.columns:
#         lc = c.lower()
#         if all(m.lower() in lc for m in must) and (any_of is None or any(a.lower() in lc for a in any_of)):
#             return c
#     return None

# def _pick_finance(site: str) -> pd.DataFrame:
#     var = "finance_phx" if site=="Phoenix" else "finance_fb"
#     if var in globals(): return globals()[var].copy()
#     if "RESULTS" in globals() and site in RESULTS and "finance" in RESULTS[site]:
#         return RESULTS[site]["finance"].copy()
#     if "finance" in globals():
#         df = globals()["finance"].copy()
#         site_col = _find_col(df, "site") or _find_col(df, "city")
#         if site_col and df[site_col].astype(str).str.contains(site, case=False, na=False).any():
#             return df
#     raise RuntimeError(f"{site}: finance table not found in memory.")

# def _extract_points(fin_tbl: pd.DataFrame) -> dict:
#     cmp_col  = _find_col(fin_tbl, "case") or _find_col(fin_tbl, "comparison") or fin_tbl.columns[0]
#     prem_col = _find_col(fin_tbl, "premium","capex") or _find_col(fin_tbl, "premium", any_of=["$","capex"])
#     spb_col  = _find_col(fin_tbl, "payback") or _find_col(fin_tbl, "simple", "payback")
#     npv_col  = _find_col(fin_tbl, "npv")  # optional
#     if prem_col is None or spb_col is None:
#         raise RuntimeError("Missing Premium CapEx or Payback column in finance table.")

#     mask = fin_tbl[cmp_col].astype(str).str.contains(r"Mag vs Cen|ATES\+?Mag vs Cen", case=False, regex=True)
#     sub  = fin_tbl.loc[mask, [cmp_col, prem_col, spb_col] + ([npv_col] if npv_col else [])].copy()

#     def _label(s): return "ATES+Mag vs Cen" if re.search(r"ates\+?mag", str(s), re.I) else "Mag vs Cen"
#     sub["Label"] = sub[cmp_col].map(_label)

#     for c in [prem_col, spb_col] + ([npv_col] if npv_col else []):
#         if c: sub[c] = pd.to_numeric(sub[c], errors="coerce")

#     pts = {}
#     for _, r in sub.iterrows():
#         lab = r["Label"]
#         pts[lab] = {
#             "spb"    : float(r[spb_col]) if np.isfinite(r[spb_col]) else np.nan,     # years
#             "premium": float(r[prem_col]) if np.isfinite(r[prem_col]) else np.nan,   # $
#             "npv"    : (float(r[npv_col]) if (npv_col and np.isfinite(r[npv_col])) else np.nan)  # $
#         }
#     return pts

# def _nice_limits(vals, pad=0.20, minimum_span=1e-6):
#     v = np.array([z for z in vals if np.isfinite(z)])
#     if v.size == 0:
#         return (0.0, 1.0)
#     lo, hi = float(v.min()), float(v.max())
#     if hi - lo < minimum_span:
#         d = max(0.1, abs(hi)*pad)
#         return (lo - d, hi + d)
#     d = (hi - lo) * pad
#     return (lo - d, hi + d)

# def plot_dynamic_with_boxes(site: str, pts: dict, savepath: str, show_boxes: bool = True):
#     names_all = [n for n in ["Mag vs Cen", "ATES+Mag vs Cen"] if n in pts]
#     x_pay   = np.array([pts[n]["spb"]                    for n in names_all], dtype=float)
#     y_capM  = np.array([pts[n]["premium"]/1e6            for n in names_all], dtype=float)  # M$
#     y_npvM  = np.array([pts[n]["npv"]/1e6                for n in names_all], dtype=float)  # M$

#     #fig, axL = plt.subplots(figsize=FIGSIZE)
#      # ---------- diagnostics ----------
#     print(f"\n[{site}] extracted points (before filtering):")
#     for n, xi, yi, vi in zip(names_all, x_pay, y_capM, y_npvM):
#         print(f"  {n:15s}  SPB={xi}  Premium(M$)={yi}  NPV(M$)={vi}")

#     # Keep only rows with finite payback AND premium (capex circles need both)
#     mask_cap = np.isfinite(x_pay) & np.isfinite(y_capM)
#     names_cap = [n for n, m in zip(names_all, mask_cap) if m]
#     X_cap = x_pay[mask_cap]
#     Y_cap = y_capM[mask_cap]
#     C_cap = [COLORS[n] for n in names_cap]

#     # NPV line can be plotted if both x and NPV are finite
#     mask_npv = np.isfinite(x_pay) & np.isfinite(y_npvM)
#     X_npv = x_pay[mask_npv]
#     Y_npv = y_npvM[mask_npv]

#     if len(names_cap) == 0:
#         print(f"[warn] {site}: no finite (SPB, Premium) pairs — circles will not be shown.")
#     else:
#         print(f"[{site}] circles to plot: {list(zip(names_cap, X_cap, Y_cap))}")

#     fig, axL = plt.subplots(figsize=FIGSIZE)

#     # Circles: Premium CapEx (M$)
#     if len(names_cap) > 0:
#         axL.scatter(
#             X_cap, Y_cap,
#             s=CAPEX_MARKER_SIZE,
#             c=C_cap,
#             edgecolor="black", linewidth=1.0,
#             zorder=5,           # <-- force on top
#             clip_on=False       # <-- prevents being clipped at axes edge
#         )

#     # Axis limits use what we actually plot (fallback to all if none valid)
#     def _safe(vals, fallback):
#         v = vals[np.isfinite(vals)]
#         return (v if v.size > 0 else fallback)

#     axL.set_xlim(_nice_limits(_safe(X_cap, x_pay), 0.22))
#     axL.set_ylim(_nice_limits(_safe(Y_cap, y_capM), 0.28))
#     axL.set_xlabel("Payback period (years)")
#     axL.set_ylabel("Premium CapEx (M$)")
#     axL.grid(True, linestyle="--", alpha=0.35)
#     axL.set_axisbelow(True)

#     # Diamonds: NPV (M$)
#     axR = axL.twinx()
#     if np.isfinite(Y_npv).any():
#         axR.plot(
#             X_npv, Y_npv,
#             ls="--", marker="D",
#             markersize=NPV_MARKER_SIZE,
#             lw=1.6, color="#6C757D", label="NPV (M$)",
#             zorder=3, clip_on=False
#         )
#         axR.set_ylim(_nice_limits(_safe(Y_npv, y_npvM), 0.28))
#     axR.set_ylabel("NPV (M$)")

#     # Legend (only include labels we actually plotted)
#     handles = []
#     for n in names_cap:
#         handles.append(Line2D([0],[0], marker='o', color='w',
#                               markerfacecolor=COLORS[n], markeredgecolor="black",
#                               markersize=9, label=n))
#     if np.isfinite(Y_npv).any():
#         handles.append(Line2D([0],[0], ls='--', marker='D', color="#6C757D",
#                               markersize=NPV_MARKER_SIZE, label="NPV (M$)"))
#     if handles:
#         axL.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1.10),
#                    ncol=max(2, len(handles)), frameon=False)


#     # -------------------- OPTIONAL GRAY BOXES (clean, deterministic) --------------------
#     if show_boxes:
#         # 1) —— BUILD TEXT CONTENT from the data (SPB, Premium, NPV) ————————————————
#         #    (this is the part that decides what goes inside the gray box)
#         labels_txt = []
#         for n, xi, yi, vi in zip(names_all, x_pay, y_capM, y_npvM):
#             line = [f"{n}", f"SPB={xi:.1f} yr", f"Premium=${pts[n]['premium']:,.0f}"]
#             if np.isfinite(vi):  # only if NPV available
#                 line.append(f"NPV={vi:.2f} M$")
#             labels_txt.append("\n".join(line))

#         # 2) —— CHOOSE BOX LOCATIONS (relative offsets per point + clamping) ——————————
#         #    (this is the part that decides where the box sits on the graph)
#         xlo, xhi = axL.get_xlim(); ylo, yhi = axL.get_ylim()
#         dx = 1.10 * (xhi + xlo)     # horizontal shift from the point
#         dy = 1.12 * (yhi + ylo)     # vertical shift from the point
#         # Simple, readable separation: first box down-left, second up-right (if both exist)
#         offsets = [(-dx, -dy), (+dx, +dy)]
#         for i, (n, xi, yi, txt) in enumerate(zip(names_all, x_pay, y_capM, labels_txt)):
#             ox, oy = offsets[i % len(offsets)]
#             tx, ty = xi + ox, yi + oy
#             # clamp inside axes with a small margin
#             mx = 0.2 * (xhi + xlo); my = 0.04 * (yhi + ylo)
#             tx = min(max(tx, xlo + mx), xhi - mx)
#             ty = min(max(ty, ylo + my), yhi - my)

#             # 3) —— DRAW THE GRAY BOX & ARROW STYLE ————————————————————————————————
#             #    (this is the part that sets colors, rounded box, and connector)
#             axL.annotate(
#                 txt, xy=(xi, yi), xytext=(tx, ty), textcoords="data",
#                 fontsize=9,
#                 bbox=dict(boxstyle="round,pad=0.25", fc="#EEEEEE", ec="0.65"),
#                 arrowprops=dict(arrowstyle="-", color="0.45", lw=0.9, connectionstyle="arc3,rad=0.1"),
#                 ha=("left" if tx<=xi else "right"),
#                 va=("bottom" if ty<=yi else "top"),
#                 zorder=4
#             )
#     # -------------------------------------------------------------------------------

#     # Bottom caption (consistent with earlier figs)
#     fig.text(0.5, 0.02, f"{site} — Dynamic payback by configuration (relative to baseline; both y- axis in M$)",
#              ha="center", va="bottom", fontsize=10)

#     plt.tight_layout(rect=[0.05, 0.07, 0.98, 0.90])
#     fig.savefig(savepath, dpi=300, bbox_inches="tight")
#     plt.close(fig)
#     print(f"[saved] {savepath}")

# # ----- build and plot both sites -----
# fin_phx = _pick_finance("Phoenix")
# fin_fb  = _pick_finance("Fairbanks")
# pts_phx = _extract_points(fin_phx)
# pts_fb  = _extract_points(fin_fb)

# # Toggle show_boxes=True/False as you like:
# plot_dynamic_with_boxes("Phoenix",   pts_phx, os.path.join(FIG_DIR, "Fig4_6_Phoenix_DynamicPayback.png"), show_boxes=True)
# plot_dynamic_with_boxes("Fairbanks", pts_fb,  os.path.join(FIG_DIR, "Fig4_7_Fairbanks_DynamicPayback.png"), show_boxes=True)


In [12]:
# ======================================================================
# Figures 4.6 & 4.7 — Dynamic payback (relative to baseline; both y-axes in M$)
# Uses fresh finance tables generated by Fin model with ATES.ipynb.
# Mag vs Cen   /   MBC+ATES vs Cen
# ======================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

FIG_DIR = r"C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures"
os.makedirs(FIG_DIR, exist_ok=True)

FIGSIZE = (9.0, 4.8)

CAPEX_MARKER_SIZE = 110
NPV_MARKER_SIZE   = 4

COLORS = {
    "Mag vs Cen"      : "#90BE6D",  # green
    "MBC+ATES vs Cen" : "#4DABF7",  # blue
}
NPV_COLOR = "#6C757D"

# ----------------------------------------------------------------------
# Build data from the latest finance tables
# Premium in $, NPV in M$, SPB in years
# ----------------------------------------------------------------------
def _finance_to_dynamic_points(finance_df: pd.DataFrame) -> dict:
    """Build dynamic-payback plot inputs from fresh finance_compare() output."""
    required = ["Case", "Premium CapEx $", "NPV of savings $", "Simple Payback (yrs)"]
    missing = [col for col in required if col not in finance_df.columns]
    if missing:
        raise RuntimeError(f"Finance table missing columns needed for dynamic plot: {missing}")

    points = {}
    for _, row in finance_df.iterrows():
        case = str(row["Case"])
        label = "MBC+ATES vs Cen" if "ATES" in case.upper() else "Mag vs Cen"
        points[label] = {
            "spb": float(row["Simple Payback (yrs)"]),
            "premium": float(row["Premium CapEx $"]),
            "npv": float(row["NPV of savings $"]) / 1e6,
        }
    return points

DATA = {
    "Phoenix": _finance_to_dynamic_points(finance_phx),
    "Fairbanks": _finance_to_dynamic_points(finance_fb),
}

def _nice_limits(vals, pad=0.15):
    v = np.array(vals, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return (0.0, 1.0)
    lo, hi = float(v.min()), float(v.max())
    if hi == lo:
        d = max(0.1, abs(hi) * pad)
        return lo - d, hi + d
    d = (hi - lo) * pad
    return lo - d, hi + d

def plot_dynamic(site: str, savepath: str):
    pts = DATA[site]
    labels = ["Mag vs Cen", "MBC+ATES vs Cen"]

    spb   = np.array([pts[l]["spb"]      for l in labels], dtype=float)
    cap_M = np.array([pts[l]["premium"]  for l in labels], dtype=float) / 1e6  # M$
    npv_M = np.array([pts[l]["npv"]      for l in labels], dtype=float)        # M$

    fig, axL = plt.subplots(figsize=FIGSIZE)

    # ----- scatter circles for Premium CapEx -----
    for x, y, lab in zip(spb, cap_M, labels):
        axL.scatter(x, y,
                    s=CAPEX_MARKER_SIZE,
                    c=COLORS[lab],
                    edgecolor="black",
                    linewidth=1.0,
                    zorder=5)

    axL.set_xlabel("Payback period (years)")
    axL.set_ylabel("Premium CapEx (M$)")
    axL.grid(linestyle="--", alpha=0.35)
    axL.set_axisbelow(True)

    xlo, xhi = _nice_limits(spb, pad=0.20)
    ylo, yhi = _nice_limits(cap_M, pad=0.25)
    axL.set_xlim(xlo, xhi)
    axL.set_ylim(ylo, yhi)

    # ----- NPV line on right axis -----
    axR = axL.twinx()
    axR.plot(spb, npv_M,
             ls="--", marker="D",
             markersize=NPV_MARKER_SIZE,
             lw=1.6, color=NPV_COLOR,
             zorder=3, clip_on=False)
    axR.set_ylabel("Discounted value of savings (M$)")
    nylo, nyhi = _nice_limits(npv_M, pad=0.25)
    axR.set_ylim(nylo, nyhi)

    # ----- legend -----
    handles = [
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor=COLORS["Mag vs Cen"], markeredgecolor="black",
               markersize=9, label="Mag vs Cen"),
        Line2D([0],[0], marker='o', color='w',
               markerfacecolor=COLORS["MBC+ATES vs Cen"], markeredgecolor="black",
               markersize=9, label="MBC+ATES vs Cen"),
        Line2D([0],[0], ls='--', marker='D', color=NPV_COLOR,
               markersize=NPV_MARKER_SIZE, label="NPV (M$)")
    ]
    axL.legend(handles=handles,
               loc="upper center",
               bbox_to_anchor=(0.5, 1.10),
               ncol=3,
               frameon=False)

               # ----- GRAY ANNOTATION BOXES (MANUAL OFFSETS) --------------------
    # You can tune these four values to move each box separately.
    # Units: DX = years (x-axis), DY = M$ (y-axis)

    DX_LOW  = -0.1   # horizontal shift for LOWER SPB point  (right is +, left is -)
    DY_LOW  = 0.50  # vertical shift   for LOWER SPB point  (up is +, down is -)

    DX_HIGH = 0.1  # horizontal shift for HIGHER SPB point
    DY_HIGH = -0.40  # vertical shift   for HIGHER SPB point

    # figure limits (for optional clamping)
    xlo, xhi = axL.get_xlim()
    ylo, yhi = axL.get_ylim()
    margin_x = 0.02 * (xhi - xlo)
    margin_y = 0.02 * (yhi - ylo)

    # index of lower and higher SPB
    idx_low  = int(np.argmin(spb))   # smaller payback
    idx_high = int(np.argmax(spb))   # larger payback

    # ---- helper to draw ONE box ----
    def _annot_one(i, dx, dy):
        lab = labels[i]
        x   = spb[i]
        y   = cap_M[i]

        txt = (
            f"{lab}\n"
            f"SPB={pts[lab]['spb']:.1f} yr\n"
            f"Premium=${pts[lab]['premium']:,.0f}\n"
            f"PV_Sav={pts[lab]['npv']:.2f} M$"
        )

        tx = x + dx
        ty = y + dy

        # optional clamp so it doesn't go completely out of frame
        tx = max(xlo + margin_x, min(tx, xhi - margin_x))
        ty = max(ylo + margin_y, min(ty, yhi - margin_y))

        axL.annotate(
            txt, xy=(x, y), xytext=(tx, ty),
            textcoords="data",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.25", fc="#EEEEEE", ec="0.65"),
            arrowprops=dict(
                arrowstyle="-",
                color="0.45",
                lw=0.9,
                connectionstyle="arc3,rad=0.1",
            ),
            ha="right" if tx >= x else "left",
            va="top" if ty >= y else "bottom",
            zorder=10,
        )

    # draw the two boxes
    _annot_one(idx_low,  DX_LOW,  DY_LOW)
    _annot_one(idx_high, DX_HIGH, DY_HIGH)


    # ----- caption -----

    plt.tight_layout(rect=[0.05, 0.07, 0.98, 0.90])
    fig.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[saved] {savepath}")

# ----------------------------------------------------------------------
# Generate Fig 4.6 and 4.7
# ----------------------------------------------------------------------
#plot_dynamic("Phoenix",   os.path.join(FIG_DIR, "Fig4_6_Phoenix_DynamicPayback.png"))
plot_dynamic("Fairbanks", os.path.join(FIG_DIR, "Fig4_7_Fairbanks_DynamicPayback.png"))


[saved] C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures\Fig4_7_Fairbanks_DynamicPayback.png


In [9]:
# === Fig 4.8 — Total cost stack (Premium CapEx + OpEx) for Phoenix & Fairbanks (values straight from vars) ===
# Reads ONLY these variables from memory (must exist after your city runs):
#   Phoenix:   capex_cen_phx / capex_cenPhoenix / capex_cen,  capex_mag_phx / capex_mag / capx_mag,  capex_amg_phx / capex_amg
#              opex_cen_phx / opex_cenPhoenix / opex_cen,    opex_mag_phx / opex_mag,               opex_amg_phx / opex_amg
#   Fairbanks: same with *_fb / *_Fairbanks fallbacks
#
# Output: ".../Result Figures/Fig4_TotalCostStack.png"

import os
import re
import numpy as np
import matplotlib.pyplot as plt

FIG_DIR = r"C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures"
os.makedirs(FIG_DIR, exist_ok=True)

# --- helper to fetch a variable by trying multiple possible names ---
def _get_var(possible_names):
    g = globals()
    for name in possible_names:
        if name in g and g[name] is not None:
            return float(g[name])
    raise RuntimeError(f"None of these variables were found in memory: {possible_names}")

# --- collect values for each site (all in $; we’ll convert to M$ on the plot) ---
def _pull_site(site):
    site_lc = site.lower()
    suf = ("_phx" if site == "Phoenix" else "_fb")
    alt = ("Phoenix" if site == "Phoenix" else "Fairbanks")

    capex_cen = _get_var([f"capex_cen{suf}", f"capex_cen_{alt}", "capex_cen"])
    # mag can be spelled capex_mag or capx_mag
    capex_mag = _get_var([f"capex_mag{suf}", f"capex_mag_{alt}", "capex_mag", f"capx_mag{suf}", f"capx_mag_{alt}", "capx_mag"])
    capex_amg = _get_var([f"capex_amg{suf}", f"capex_amg_{alt}", "capex_amg"])

    opex_cen  = _get_var([f"opex_cen{suf}",  f"opex_cen_{alt}",  "opex_cen"])
    opex_mag  = _get_var([f"opex_mag{suf}",  f"opex_mag_{alt}",  "opex_mag"])
    opex_amg  = _get_var([f"opex_amg{suf}",  f"opex_amg_{alt}",  "opex_amg"])

    return {
        "Centrifugal": {"CapEx": capex_cen, "OpEx": opex_cen},
        "Magnetic":    {"CapEx": capex_mag, "OpEx": opex_mag},
        "MBC+ATES":    {"CapEx": capex_amg, "OpEx": opex_amg},
    }

data_phx = _pull_site("Phoenix")
data_fb  = _pull_site("Fairbanks")

# --- plotting (consistent style; same color per plan; CapEx + OpEx stacked; numbers on segments + totals) ---
PLAN_ORDER = ["Centrifugal", "Magnetic", "MBC+ATES"]
COLORS = {
    "Centrifugal": "#F9C74F",  # yellow
    "Magnetic":    "#90BE6D",  # green
    "MBC+ATES":    "#4DABF7",  # blue
}

def _plot_stack(ax, city_name, city_data, xoffset):
    width = 0.20
    xs = np.array([xoffset + i*width*1.3 for i in range(len(PLAN_ORDER))])  # small spacing within city
    capex = np.array([city_data[p]["CapEx"] for p in PLAN_ORDER], dtype=float) / 1e6  # -> M$
    opex  = np.array([city_data[p]["OpEx"]  for p in PLAN_ORDER], dtype=float) / 1e6  # -> M$
    total = capex + opex

    # CapEx bars (bottom) and OpEx bars (top) — SAME color per plan
    for i, p in enumerate(PLAN_ORDER):
        ax.bar(xs[i], capex[i], width=width, color=COLORS[p], edgecolor="black", alpha=0.45)  # CapEx lighter
        ax.bar(xs[i], opex[i],  width=width, bottom=capex[i], color=COLORS[p], edgecolor="black", alpha=0.95)  # OpEx solid

        # Segment labels (CapEx/OpEx)
        ax.text(xs[i], capex[i]/2, f"{capex[i]:.2f}", ha="center", va="center", fontsize=9, color="black")
        ax.text(xs[i], capex[i] + opex[i]/2, f"{opex[i]:.2f}", ha="center", va="center", fontsize=9, color="black")

        # Total on top
        ax.text(xs[i], total[i] + (0.02*max(1.0,total.max())), f"{total[i]:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

    # City label under the cluster
    ax.text(xs.mean(), -0.06*max(1.0,(capex+opex).max()), city_name, ha="center", va="top", fontsize=11)

    return xs

fig, ax = plt.subplots(figsize=(9.5, 5.0))

# Phoenix cluster centered at x ~ 0, Fairbanks ~ 1
xs_phx = _plot_stack(ax, "Phoenix",   data_phx, xoffset=0.0)
xs_fb  = _plot_stack(ax, "Fairbanks", data_fb,  xoffset=1.2)

# Cosmetics
ax.set_ylabel("Premium CapEx + OpEx (M$)")
ax.set_xticks([])  # city names already drawn under clusters
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

# === Custom Y-axis scale control ===
ax.set_ylim(0, 1.2 * max(1.0, max([d["CapEx"] + d["OpEx"] for dat in [data_phx, data_fb] for d in dat.values()]) / 1e6))
ax.set_yticks(np.arange(0, ax.get_ylim()[1] + 0.001, 0.3))  # steps of 0.3 M$


# Legend — one entry per plan (same color used for both segments)
legend_handles = [plt.Line2D([0],[0], marker='s', color='w', markerfacecolor=COLORS[p],
                             markeredgecolor="black", markersize=10, label=p) for p in PLAN_ORDER]
ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=3, frameon=False)

# Bottom caption (consistent with earlier figs)
fig.text(0.5, 0.02, "Bars show Premium CapEx + OpEx (energy + O&M).",
         ha="center", va="bottom", fontsize=10)

plt.tight_layout(rect=[0.04, 0.07, 0.98, 0.93])
savepath = os.path.join(FIG_DIR, "Fig4_TotalCostStack.png")
fig.savefig(savepath, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[saved] {savepath}")


[saved] C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures\Fig4_TotalCostStack.png


In [10]:
# === Fig 4.X — Premium CapEx + OpEx with Simple Payback overlay (both cities) ===
# Uses FINAL numbers read from your Phoenix/Fairbanks tables.
# Legend text: "MBC+ATES" (was ATES+Mag).

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os

# ---------- paths & figure size ----------
FIG_DIR = r"C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures"
os.makedirs(FIG_DIR, exist_ok=True)
FIGSIZE   = (10, 6.5)
SAVE_PATH = os.path.join(FIG_DIR, "Fig4_X_Cost_With_SPB.png")

# -------------------------------------------------------------------
# 1) Values from fresh finance/model outputs, not rounded old figures.
# -------------------------------------------------------------------
required_cost_vars = [
    "capex_cen_phx", "capex_mag_phx", "capex_amg_phx", "opex_cen_phx", "opex_mag_phx", "opex_amg_phx",
    "capex_cen_fb", "capex_mag_fb", "capex_amg_fb", "opex_cen_fb", "opex_mag_fb", "opex_amg_fb",
    "spb_mag_phx", "spb_amg_phx", "spb_mag_fb", "spb_amg_fb",
]
missing_cost_vars = [name for name in required_cost_vars if name not in globals()]
if missing_cost_vars:
    raise RuntimeError(f"Run Fin model with ATES.ipynb first; missing: {missing_cost_vars}")

capex_mbcates_phx = capex_amg_phx
opex_mbcates_phx  = opex_amg_phx
spb_mbcates_phx   = spb_amg_phx

capex_mbcates_fb  = capex_amg_fb
opex_mbcates_fb   = opex_amg_fb
spb_mbcates_fb    = spb_amg_fb

# -------------------------------------------------------------------
# 2) ARRAYS (convert to M$ here)
# -------------------------------------------------------------------
PLANS = ["Centrifugal", "Magnetic", "MBC+ATES"]
CLR   = {
    "Centrifugal": "#F9C74F",  # yellow
    "Magnetic":    "#90BE6D",  # green
    "MBC+ATES":    "#4DABF7",  # blue
}

def _M(usd):
    return np.array(usd, dtype=float) / 1e6

# Phoenix (M$)
cap_P = _M([capex_cen_phx,     capex_mag_phx,     capex_mbcates_phx])
ope_P = _M([opex_cen_phx,      opex_mag_phx,      opex_mbcates_phx])

# Fairbanks (M$)
cap_F = _M([capex_cen_fb,      capex_mag_fb,      capex_mbcates_fb])
ope_F = _M([opex_cen_fb,       opex_mag_fb,       opex_mbcates_fb])

tot_P = cap_P + ope_P
tot_F = cap_F + ope_F

spb_P = np.array([0.0, spb_mag_phx,    spb_mbcates_phx], dtype=float)
spb_F = np.array([0.0, spb_mag_fb,     spb_mbcates_fb],  dtype=float)

# -------------------------------------------------------------------
# 3) PLOT
# -------------------------------------------------------------------
fig, axL = plt.subplots(figsize=FIGSIZE)

group_gap = 0.8
bar_width = 0.4

gx  = np.array([0.0, 1.0 + group_gap])       # group centers: Phoenix, Fairbanks
off = np.array([-bar_width, 0.0, +bar_width]) * 0.6

xP = gx[0] + off
xF = gx[1] + off

# Phoenix bars
for i, p in enumerate(PLANS):
    axL.bar(xP[i], cap_P[i], width=bar_width, color=CLR[p],
            edgecolor="black", linewidth=1.0, alpha=0.95, zorder=2)
    axL.bar(xP[i], ope_P[i], width=bar_width, bottom=cap_P[i],
            color=CLR[p], edgecolor="black",
            linewidth=1.0, alpha=0.35, zorder=2)

# Fairbanks bars
for i, p in enumerate(PLANS):
    axL.bar(xF[i], cap_F[i], width=bar_width, color=CLR[p],
            edgecolor="black", linewidth=1.0, alpha=0.95, zorder=2)
    axL.bar(xF[i], ope_F[i], width=bar_width, bottom=cap_F[i],
            color=CLR[p], edgecolor="black",
            linewidth=1.0, alpha=0.35, zorder=2)

# --- segment labels (inside bar if tall enough) ---
def label_segment(ax, x, bottom, height, value, *, min_h=0.06, pad=0.02):
    if height <= 0:
        return
    if height >= min_h:
        y  = bottom + 0.5 * height
        va = "center"
    else:
        y  = bottom + height + pad
        va = "bottom"
    ax.text(x, y, f"{value:.2f}", ha="right", va=va, fontsize=9, color="black")

# Phoenix labels
for i in range(3):
    label_segment(axL, xP[i], 0.0,      cap_P[i], cap_P[i])
    label_segment(axL, xP[i], cap_P[i], ope_P[i], ope_P[i])

# Fairbanks labels
for i in range(3):
    label_segment(axL, xF[i], 0.0,      cap_F[i], cap_F[i])
    label_segment(axL, xF[i], cap_F[i], ope_F[i], ope_F[i])

# Totals above bars
def label_total(ax, x, t):
    ax.text(x, t + 0.15, f"{t:.2f}", ha="right", va="bottom",
            fontsize=10, fontweight="bold")

for i in range(3):
    label_total(axL, xP[i], tot_P[i])
    label_total(axL, xF[i], tot_F[i])

axL.set_ylabel("Premium CapEx + OpEx (M$)")
axL.set_xticks([gx[0], gx[1]])
axL.set_xticklabels(["Phoenix", "Fairbanks"])
axL.grid(axis="y", linestyle="--", alpha=0.35)
axL.set_axisbelow(True)

ymaxM = max(np.max(tot_P), np.max(tot_F))
axL.set_ylim(0, ymaxM * 1.18)
axL.set_yticks(np.arange(0, axL.get_ylim()[1] + 1e-9, 0.3))

# ---------- overlay: simple payback ----------
axR = axL.twinx()
axR.set_ylabel("Simple payback (years)")

xP_upg   = [xP[1], xP[2]]
xF_upg   = [xF[1], xF[2]]
spbP_upg = [spb_P[1], spb_P[2]]
spbF_upg = [spb_F[1], spb_F[2]]

axR.scatter(xP_upg, spbP_upg, color="#6C757D", edgecolor="black", s=70,
            label="SPB (Phoenix)", zorder=3)
axR.scatter(xF_upg, spbF_upg, facecolors="#6C757D", edgecolor="red", s=70,
            label="SPB (Fairbanks)", zorder=3)

def nice_limits(vals, pad=0.25):
    v = np.array([z for z in vals if np.isfinite(z)])
    if v.size == 0:
        return (0, 1)
    lo, hi = float(np.nanmin(v)), float(np.nanmax(v))
    d = max(0.6, (hi - lo) * pad)
    return (max(0.0, lo - 0.2*d), hi + d)

axR.set_ylim(*nice_limits(np.r_[spb_P, spb_F]))

# Legends
plan_handles = [
    Line2D([0],[0], marker='s', color='w',
           markerfacecolor=CLR[p], markeredgecolor="black",
           markersize=10, label=p)
    for p in PLANS
]
axL.legend(handles=plan_handles, loc="upper center",
           bbox_to_anchor=(0.5, 1.08), ncol=3, frameon=False)

spb_handles, spb_labels = axR.get_legend_handles_labels()
axR.legend(spb_handles, spb_labels, loc="upper right", frameon=False)

fig.text(0.5, 0.02,
         "Bars: Premium CapEx (solid) + OpEx (transparent) in M$. Lines: simple payback (years).",
         ha="center", va="bottom", fontsize=10)

plt.tight_layout(rect=[0.05, 0.07, 0.98, 0.90])
fig.savefig(SAVE_PATH, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[saved] {SAVE_PATH}")


[saved] C:\Users\apurv\OneDrive\Desktop\THESIS\Financial Model .pynb files\Result Figures\Fig4_X_Cost_With_SPB.png


In [ ]:
# Axis labels are now set in the plotting cells.
# Do not overlay extra text on saved PNG files.
print("Axis labels are handled in the plotting cells.")
